# Part D: Cross-Validation Strategies
**Robust Regression Engine**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut,
    TimeSeriesSplit, cross_val_score, train_test_split
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")

TARGET    = 'house_price_inr'
DROP_COLS = ['property_id', 'sale_date']
FEATURES  = [col for col in df.columns if col not in [TARGET] + DROP_COLS]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

model = Ridge(alpha=1000)

print("Data Ready")
print(X_train.shape)

## Task 13 — K-Fold Cross-Validation

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=kf,
    scoring='r2'
)

print("Cross Validation Scores :", cv_scores)
print("Average CV Score        :", cv_scores.mean())

## Task 13 — Stratified K-Fold Cross-Validation

In [ ]:
# Create charge category for stratification
y_bins = pd.qcut(y_train, q=5, labels=False, duplicates='drop')

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    model,
    X_train_scaled,
    y_train,
    cv=list(skf.split(X_train_scaled, y_bins)),
    scoring='r2'
)

print("Scores  :", scores)
print("Average :", scores.mean())

## Task 13 — Leave-One-Out Cross-Validation (LOOCV)

In [ ]:
# Using subset of 200 rows for speed (LOOCV runs N iterations)
X_sub = X_train_scaled[:200]
y_sub = y_train.values[:200]

loo = LeaveOneOut()

loo_scores = cross_val_score(
    model,
    X_sub,
    y_sub,
    cv=loo,
    scoring='neg_mean_squared_error'
)

print("Cross Validation Scores :", loo_scores[:10], "...")
print("Mean MSE :", abs(loo_scores.mean()))

## Task 13 — Time Series Split (sorted by sale_date)

In [ ]:
df_ts = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")
df_ts['sale_date'] = pd.to_datetime(df_ts['sale_date'])
df_ts = df_ts.sort_values('sale_date').reset_index(drop=True)

X_ts = df_ts[FEATURES].values
y_ts = df_ts[TARGET].values
X_ts_scaled = StandardScaler().fit_transform(X_ts)

tscv = TimeSeriesSplit(n_splits=5)

ts_scores = cross_val_score(
    model,
    X_ts_scaled,
    y_ts,
    cv=tscv,
    scoring='r2'
)

print("Scores       :", ts_scores)
print("Average Score:", ts_scores.mean())

## Task 14 — Analyze Performance Across CV Strategies

In [ ]:
results = {
    'K-Fold':            cv_scores,
    'Stratified K-Fold': scores,
    'Time Series Split': ts_scores
}

# Boxplot
plt.figure(figsize=(9, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel('R2 Score')
plt.title('R2 Distribution across Cross-Validation Strategies')
plt.show()

print("\nSummary:")
for name, s in results.items():
    print(f"  {name:22s}  Mean={s.mean():.4f}  Std={s.std():.4f}")
print(f"  {'LOOCV (n=200)':22s}  Mean MSE={abs(loo_scores.mean()):.2f}")